# Extracting morphometric information from segmented grains
### 
- This notebook is intended to be run after all grains have been properly segmented, using either the [create_polygons_and_masks.ipynb](https://github.com/owachob/segmenteveryzircon/blob/main/1_create_polygons_and_masks.ipynb) file or otherwise.

Importing necesary libraries

In [9]:
from importlib import reload
from keras.saving import load_model
from keras.utils import load_img
from PIL import Image
import cv2
import geopandas as gpd
import numpy as np
import os
import pandas as pd
import rasterio
from rasterio.features import rasterize
import seaborn as sns
import segmenteverygrain as seg
import sez
from shapely import wkt
from shapely.geometry import Polygon
from skimage.measure import regionprops, regionprops_table
import tifffile
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib qt
import matplotlib.image as mpimg
from segment_anything import SamPredictor, sam_model_registry

### Step 1: Read in the original image

In [2]:
original_image_path = 'example_images/example_input_image.png'
original_image = cv2.imread('example_images/example_input_image.png')

### Step 2: Read in the Csv file with the coordinates of the segmented polygon to re-initalize the geodataframe

In [3]:
# Replace with path to your csv file of the segmented polygon coordinates
csv_path = '/Users/omw339/Desktop/example_coordinates.csv'

In [4]:
height, width = original_image.shape[:2]

# Create shapes generator: (geometry, label) tuples
gdf = sez.load_polygons(csv_path, crs="EPSG:4326")
shapes = ((geom, idx + 1) for idx, geom in enumerate(gdf.geometry))

# Rasterize polygons:
label_image = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    fill=0,
    dtype=np.uint16
)
all_grains = list(gdf.geometry)
print(f"Rasterized label image shape: {label_image.shape}")
print(f"Max label value (should equal number of polygons): {label_image.max()}")

   Unnamed: 0                                           geometry
0           0  POLYGON ((1250.5 129, 1249.5 129, 1248.5 129, ...
1           1  POLYGON ((1703.5 147, 1702.5 147, 1701.5 147, ...
2           2  POLYGON ((943.5 178, 942.5 178, 941.5 178, 940...
3           3  POLYGON ((1011.5 238, 1010.5 238, 1009.5 238, ...
4           4  POLYGON ((684.5 228, 683.5 228, 682.5 228, 681...
Number of polygons loaded: 109
Rasterized label image shape: (1447, 2202)
Max label value (should equal number of polygons): 109


Ensure that the max label value is equivalent to the number of grains you segmented, otherwise double-check your csv file

In [5]:
tifffile.imwrite('/Users/omw339/Desktop/example_rasterized_labels.tif', label_image)

### Step 3: Creating the morphometrics dataframe.
- The following cell creates the dataframe for the morphometric information. It is important to note that the original columns are in **pixels**

In [6]:
from skimage.measure import regionprops_table

# Define the properties you want
properties = [
    'label',
    'area',
    'centroid',
    'bbox',
    'major_axis_length',
    'minor_axis_length',
    'eccentricity',
    'solidity',
    'orientation',
    'perimeter'
]

# Extract properties for each labeled region (grain)
props = regionprops_table(label_image, properties=properties)

# Convert to DataFrame
grain_data = pd.DataFrame(props)

print(grain_data.head())


   label    area  centroid-0   centroid-1  bbox-0  bbox-1  bbox-2  bbox-3  \
0      1  5863.0   92.251407  1235.174655      51    1177     129    1285   
1      2  5007.0  102.885960  1672.041342      59    1630     147    1714   
2      3  4570.0  133.642013   950.227571     100     912     178     994   
3      4  4109.0  199.374787  1015.194938     170     975     238    1059   
4      5  3755.0  195.920107   692.466045     166     655     228     734   

   major_axis_length  minor_axis_length  eccentricity  solidity  orientation  \
0         103.137875          74.293976      0.693625  0.961463    -1.374285   
1         107.134194          59.624347      0.830822  0.983307     0.727068   
2          89.992636          67.362119      0.663102  0.979006    -0.802585   
3          82.767489          65.173490      0.616406  0.981137    -1.386516   
4          86.248075          55.805612      0.762459  0.977610    -1.165229   

    perimeter  
0  301.806133  
1  279.663997  
2  265.0

### Step 4: The following cells are used to get morphometric information in micrometers (or any other unit)

**Option 1**: Use the length of the scale bar in pixels to get the scale of the image (in units / pixel) within the notebook. Run this cell and then click (left mouse button) on one end of the scale bar in the image and click (right mouse button) on the other end of the scale bar:

In [10]:
image = np.array(load_img(original_image_path))
fig, ax = plt.subplots(figsize=(15,10))
plt.xticks([])
plt.yticks([])
seg.plot_image_w_colorful_grains(image, all_grains, ax, cmap='Paired')
plt.axis('equal')
plt.xlim([0, np.shape(image)[1]])
plt.ylim([np.shape(image)[0], 0]);

100%|██████████| 109/109 [00:00<00:00, 385.17it/s]


In [11]:
cid5 = fig.canvas.mpl_connect('button_press_event', lambda event: seg.click_for_scale(event, ax))

number of pixels: 182.18


**Option 2 (Recommended)**: Open up the original image in any image processing software such as FIJI or ImageJ to measure the scalebar

- n_of_units: represents the number on the scale bar in the image
- scale_bar_length: the length of the scale bar in pixels that you measured


In [12]:
n_of_units = 200 # micrometers usually'
scale_bar_length = 182.18 #length of scale bar in pixels
units_per_pixel = n_of_units/scale_bar_length

Adding columns to the grain_data dataframe for the morphometric measurements in the scaled measurement of your choice

In [13]:
grain_data_micron = sez.convert_grain_units(grain_data, units_per_pixel)

Adding measurements like roundness and aspect ratio. Now that you have the grain geometries, you can derive any other morphometric parameters as you see fit.

In [18]:
grain_data_micron['roundness'] = (4 * grain_data_micron['area']) / (np.pi * (grain_data_micron['major_axis_length']**2))
grain_data_micron['aspect_ratio'] = grain_data_micron['major_axis_length'] / grain_data_micron['minor_axis_length']

In [20]:
print(grain_data_micron)

     label    area  centroid-0   centroid-1  bbox-0  bbox-1  bbox-2  bbox-3  \
0        1  5863.0   92.251407  1235.174655      51    1177     129    1285   
1        2  5007.0  102.885960  1672.041342      59    1630     147    1714   
2        3  4570.0  133.642013   950.227571     100     912     178     994   
3        4  4109.0  199.374787  1015.194938     170     975     238    1059   
4        5  3755.0  195.920107   692.466045     166     655     228     734   
..     ...     ...         ...          ...     ...     ...     ...     ...   
104    105  2736.0  389.649123  1079.191155     359    1048     420    1113   
105    106  5104.0  364.978644  1138.367359     321    1104     411    1175   
106    107  4275.0  901.449825  1332.610058     866    1294     951    1366   
107    108  4408.0  831.167877  1337.024955     791    1301     865    1383   
108    109  3459.0  571.339983   354.531367     537     324     607     387   

     major_axis_length  minor_axis_length  ...  maj

saving morphometric dataframe to csv

In [21]:
grain_data.to_csv('/Users/omw339/Desktop/example_morphometrics.csv')

## Optional Visualizations

Figure 1: Scatterplot of Aspect Ratios 

In [27]:
x = grain_data_micron['minor_axis_length_micron']
y = grain_data_micron['major_axis_length_micron']

plt.scatter(x, y, c = grain_data_micron['area_micron2'], cmap='gnuplot', s=50)

plt.xlabel('Minor Axis Length (µm)', fontsize=14)
plt.ylabel('Major Axis Length (µm)', fontsize=14)
plt.title('Zircon Axes Lengths', fontsize = 20)
plt.colorbar(label = 'Area (µm^2)')

plt.show()

Figure 2: Histogram of Aspect Ratios

In [28]:
plt.figure(figsize=(8, 5))
sns.histplot(grain_data_micron['aspect_ratio'].dropna(), bins=30, kde=False, color='lightblue', edgecolor='black')
plt.title("Sample 2 Aspect Ratios")
plt.xlabel("Aspect Ratio (µm)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### Inspecting a Particular Grain (Outlier Verification)
The following cells to extract the morphometric data and shape for a particular grain ID to confirm accuracy of outliers.

In [ ]:
chosen_grain = input("Please enter a grain number from 1 to " + str(len(gdf)) + ":")
gnumupdated = chosen_grain
gnumindex = int(chosen_grain) - 1

while len(gnumupdated) !=4:
    gnumupdated = "0" + gnumupdated

Once you have selected a grain_id, the following cell prints out its associated morphometric data

In [ ]:
grain_data_specific = grain_data[grain_data["label"] == int(chosen_grain)]
print("Here is your grain's data:")
grain_data_specific

This cell produces an overlay of the segmentation (in a thin green line) over the grain from the original image

In [ ]:
sez.show_grain_overlay(chosen_grain, grain_data, label_image, original_image)